## The script below just trains on both PrimeVul and a combined Primevul and KyVul


In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_curve, roc_curve, auc, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import json
import matplotlib.pyplot as plt
import os

# Set seeds for reproducibility (will be updated per trial)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Check if GPU is available and use it if present
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Function to load and preprocess the dataset line by line
def load_and_preprocess_data_line_by_line(file_path, batch_size=500, is_jsonl=True):
    texts, labels = [], []
    with open(file_path, 'r') as file:
        if is_jsonl:
            for line in file:
                data = json.loads(line.strip())
                texts.append(data['func'])
                labels.append(data['target'])
                if len(texts) >= batch_size:
                    yield texts, labels
                    texts, labels = [], []
        else:  # Assuming JSON format for KyVul
            data = json.load(file)
            for item in data:
                texts.append(item['code'])
                labels.append(item['vulnerable'])
                if len(texts) >= batch_size:
                    yield texts, labels
                    texts, labels = [], []
    if len(texts) > 0:
        yield texts, labels

# Load the pre-trained CodeBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

# Tokenization function
def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=512, return_tensors="pt")

# Define the custom dataset class
class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]).long()
        return item

    def __len__(self):
        return len(self.labels)

# Function to process and tokenize data
def process_and_tokenize_data(file_path, batch_size=500, is_jsonl=True):
    encodings = {'input_ids': [], 'attention_mask': []}
    labels = []

    print(f"Processing data from {file_path}...")
    for batch_texts, batch_labels in load_and_preprocess_data_line_by_line(file_path, batch_size, is_jsonl=is_jsonl):
        batch_encodings = tokenize_function(batch_texts)
        for key in encodings:
            encodings[key].extend(batch_encodings[key])
        labels.extend(batch_labels)

    return encodings, labels

# Load and tokenize PrimeVul data
primevul_train_encodings, primevul_train_labels = process_and_tokenize_data('PrimeVul/primevul_train.jsonl', batch_size=500, is_jsonl=True)
primevul_valid_encodings, primevul_valid_labels = process_and_tokenize_data('PrimeVul/primevul_valid.jsonl', batch_size=500, is_jsonl=True)
primevul_test_encodings, primevul_test_labels = process_and_tokenize_data('PrimeVul/primevul_test.jsonl', batch_size=500, is_jsonl=True)

# Load and tokenize KyVul data
kyvul_train_encodings, kyvul_train_labels = process_and_tokenize_data('kyVul.json', batch_size=500, is_jsonl=False)

# Combine PrimeVul and KyVul training data
combined_train_encodings = {'input_ids': [], 'attention_mask': []}
combined_train_encodings['input_ids'].extend(primevul_train_encodings['input_ids'])
combined_train_encodings['input_ids'].extend(kyvul_train_encodings['input_ids'])
combined_train_encodings['attention_mask'].extend(primevul_train_encodings['attention_mask'])
combined_train_encodings['attention_mask'].extend(kyvul_train_encodings['attention_mask'])
combined_train_labels = primevul_train_labels + kyvul_train_labels

# Create datasets
primevul_train_dataset = CodeDataset(primevul_train_encodings, primevul_train_labels)
primevul_valid_dataset = CodeDataset(primevul_valid_encodings, primevul_valid_labels)
primevul_test_dataset = CodeDataset(primevul_test_encodings, primevul_test_labels)

combined_train_dataset = CodeDataset(combined_train_encodings, combined_train_labels)
combined_valid_dataset = CodeDataset(primevul_valid_encodings, primevul_valid_labels)  # Use PrimeVul validation for consistency

# Define the evaluation function for accuracy
def compute_metrics(p):
    preds = p.predictions.argmax(axis=1)
    labels = p.label_ids
    accuracy = accuracy_score(labels, preds)
    return {"accuracy": accuracy}

# Function to train and evaluate a model
def train_and_evaluate_model(train_dataset, valid_dataset, test_dataset, output_dir, plot_folder, trial):
    # Load a fresh model instance
    model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
    model.to(device)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir=f'./{output_dir}',
        learning_rate=1e-4,
        num_train_epochs=20,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=128,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir=f'./logs_{output_dir}',
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        fp16=True,
        report_to="none",
        disable_tqdm=False,
        log_level='info',
        
    )

    # Initialize Trainer with EarlyStoppingCallback
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # Train the model
    print(f"\nTraining model for {output_dir} (Trial {trial})...")
    trainer.train()

    # Save the best model
    model.save_pretrained(f'./{output_dir}')
    tokenizer.save_pretrained(f'./{output_dir}')

    # Evaluate on PrimeVul test set
    print(f"\nEvaluating on PrimeVul test set for {output_dir} (Trial {trial})...")
    eval_trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f'./{output_dir}',
            per_device_eval_batch_size=128,
            do_train=False,
            do_eval=True,
            logging_dir=f'./logs_{output_dir}',
            logging_steps=10,
            fp16=False,
            report_to="none",
            disable_tqdm=False,
        ),
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    # Perform evaluation
    eval_results = eval_trainer.evaluate()
    print(f"Evaluation results for {output_dir} (Trial {trial}):", eval_results)

    # Get predictions for PRC and ROC
    predictions, true_labels, _ = eval_trainer.predict(test_dataset)
    probs = torch.softmax(torch.tensor(predictions), dim=1)[:, 1].numpy()

    # Calculate precision, recall, f1-score
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions.argmax(axis=1), average='binary')
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")

    # Create the output folder for plots
    os.makedirs(plot_folder, exist_ok=True)

    # Plot Precision-Recall Curve
    precision_curve, recall_curve, _ = precision_recall_curve(true_labels, probs)
    plt.figure(figsize=(8, 6))
    plt.plot(recall_curve, precision_curve, label=f'PRC (AUC = {auc(recall_curve, precision_curve):.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {output_dir} (Trial {trial})')
    plt.legend(loc='best')
    plt.grid(True)
    plt.savefig(os.path.join(plot_folder, 'prc_curve.png'))
    plt.close()

    # Plot ROC Curve
    fpr, tpr, _ = roc_curve(true_labels, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic Curve - {output_dir} (Trial {trial})')
    plt.legend(loc='best')
    plt.grid(True)
    plt.savefig(os.path.join(plot_folder, 'roc_curve.png'))
    plt.close()

    print(f"PRC and ROC curves saved to '{plot_folder}' folder.")

    # Clear GPU memory
    torch.cuda.empty_cache()

# Run training and evaluation for 5 trials
for trial in range(1, 6):
    # Set random seed for this trial
    seed = 16 + trial
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Train and evaluate PrimeVul-only model
    train_and_evaluate_model(
        train_dataset=primevul_train_dataset,
        valid_dataset=primevul_valid_dataset,
        test_dataset=primevul_test_dataset,
        output_dir=f'PrimeVulBert_Trial{trial}',
        #plot_folder=f'BertGKyPrime/NonTokenized/PrimeVul/Trial{trial}',
        plot_folder=f'Testing/PrimeVul/Trial{trial}',
        trial=trial
    )

    # Train and evaluate Combined model
    train_and_evaluate_model(
        train_dataset=combined_train_dataset,
        valid_dataset=combined_valid_dataset,
        test_dataset=primevul_test_dataset,
        output_dir=f'KyPrimeVulBert_Trial{trial}',
        #plot_folder=f'BertGKyPrime/NonTokenized/Combined/Trial{trial}',
        plot_folder=f'Testing/NonTokenized/Trial{trial}',
        trial=trial
    )

# To get the graphs with proper sig figs, run the code below after the code above completes

In [ ]:
import torch
import random
import numpy as np
import json
import os
from sklearn.metrics import precision_recall_curve, roc_curve, auc
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import matplotlib.pyplot as plt

# Set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the pre-trained CodeBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

# Function to load and preprocess the dataset line by line
def load_and_preprocess_data_line_by_line(file_path, batch_size=500, is_jsonl=True):
    texts, labels = [], []
    with open(file_path, 'r') as file:
        if is_jsonl:
            for line in file:
                data = json.loads(line.strip())
                texts.append(data['func'])
                labels.append(data['target'])
                if len(texts) >= batch_size:
                    yield texts, labels
                    texts, labels = [], []
        else:
            data = json.load(file)
            for item in data:
                texts.append(item['code'])
                labels.append(item['vulnerable'])
                if len(texts) >= batch_size:
                    yield texts, labels
                    texts, labels = [], []
    if len(texts) > 0:
        yield texts, labels

# Tokenization function
def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=512, return_tensors="pt")

# Function to process and tokenize data
def process_and_tokenize_data(file_path, batch_size=500, is_jsonl=True):
    encodings = {'input_ids': [], 'attention_mask': []}
    labels = []
    print(f"Processing data from {file_path}...")
    for batch_texts, batch_labels in load_and_preprocess_data_line_by_line(file_path, batch_size, is_jsonl=is_jsonl):
        batch_encodings = tokenize_function(batch_texts)
        for key in encodings:
            encodings[key].extend(batch_encodings[key])
        labels.extend(batch_labels)
    return encodings, labels

# Define the custom dataset class
class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]).long()
        return item

    def __len__(self):
        return len(self.labels)

# Load and tokenize PrimeVul test data
primevul_test_encodings, primevul_test_labels = process_and_tokenize_data('PrimeVul/primevul_test.jsonl', batch_size=500, is_jsonl=True)
test_dataset = CodeDataset(primevul_test_encodings, primevul_test_labels)

# Function to evaluate a model and return probabilities
def evaluate_model(model, test_dataset):
    model.to(device)
    eval_trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir='./results',
            per_device_eval_batch_size=128,
            do_train=False,
            do_eval=True,
            logging_dir='./logs',
            logging_steps=10,
            fp16=False,
            report_to="none",
            disable_tqdm=False,
        ),
        eval_dataset=test_dataset,
    )
    predictions, true_labels, _ = eval_trainer.predict(test_dataset)
    probs = torch.softmax(torch.tensor(predictions), dim=1)[:, 1].numpy()
    return probs, true_labels

# Function to plot PRC and ROC for both models on the same figure with subplots
def plot_combined_metrics(primevul_probs, combined_probs, true_labels, plot_folder, trial):
    # Compute PRC for both models
    primevul_precision, primevul_recall, _ = precision_recall_curve(true_labels, primevul_probs)
    primevul_prc_auc = auc(primevul_recall, primevul_precision)
    combined_precision, combined_recall, _ = precision_recall_curve(true_labels, combined_probs)
    combined_prc_auc = auc(combined_recall, combined_precision)

    # Compute ROC for both models
    primevul_fpr, primevul_tpr, _ = roc_curve(true_labels, primevul_probs)
    primevul_roc_auc = auc(primevul_fpr, primevul_tpr)
    combined_fpr, combined_tpr, _ = roc_curve(true_labels, combined_probs)
    combined_roc_auc = auc(combined_fpr, combined_tpr)

    # Create a figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot PRC for both models on the left subplot
    axes[0].plot(primevul_recall, primevul_precision, label=f'PrimeVul PRC (AUC = {primevul_prc_auc:.3f})', color='blue')
    axes[0].plot(combined_recall, combined_precision, label=f'Combined PRC (AUC = {combined_prc_auc:.3f})', color='orange')
    axes[0].set_xlabel('Recall')
    axes[0].set_ylabel('Precision')
    axes[0].set_title(f'Precision-Recall Curves - Trial {trial}')
    axes[0].legend(loc='best')
    axes[0].grid(True)

    # Plot ROC for both models on the right subplot
    axes[1].plot(primevul_fpr, primevul_tpr, label=f'PrimeVul ROC (AUC = {primevul_roc_auc:.3f})', color='blue')
    axes[1].plot(combined_fpr, combined_tpr, label=f'Combined ROC (AUC = {combined_roc_auc:.3f})', color='orange')
    axes[1].plot([0, 1], [0, 1], 'k--')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curves - Trial {trial}')
    axes[1].legend(loc='best')
    axes[1].grid(True)

    # Save the combined figure
    os.makedirs(plot_folder, exist_ok=True)
    fig.savefig(os.path.join(plot_folder, 'prc_roc_combined.png'))
    plt.close(fig)

    print(f"Combined PRC and ROC plot saved to '{plot_folder}/prc_roc_combined.png'.")

# Base directory for saving plots
BASE_PLOT_DIR = './thesisGraphs/PRC'

# Reevaluate models for each trial
for trial in range(1, 6):
    # Load PrimeVul-only model
    primevul_model = AutoModelForSequenceClassification.from_pretrained(f'./PrimeVulBert_Trial{trial}')
    primevul_probs, true_labels = evaluate_model(primevul_model, test_dataset)

    # Load Combined model
    combined_model = AutoModelForSequenceClassification.from_pretrained(f'./KyPrimeVulBert_Trial{trial}')
    combined_probs, _ = evaluate_model(combined_model, test_dataset)

    # Plot combined PRC and ROC in subplots
    plot_folder = os.path.join(BASE_PLOT_DIR, f'Trial{trial}')
    plot_combined_metrics(primevul_probs, combined_probs, true_labels, plot_folder, trial)